# AIC 2026 — Pipeline Offline đầy đủ (Batch 1)

**Chạy được trên cả Google Colab lẫn máy cá nhân (Local)** — Phần 0 tự động phát hiện đang chạy ở đâu và chọn đúng đường dẫn input/output tương ứng, phần còn lại của notebook không đổi.

Notebook này thực hiện đủ 8 bước của Pipeline Offline đã thiết kế, dựa trên dữ liệu Batch 1 đã có sẵn trên Drive (`video/`, `keyframes/`, `objects*.zip`, `media-info*.zip`, `map-keyframes*.zip`, `clip-features*.zip`).

**3 nguyên tắc xuyên suốt notebook này** (rút ra từ các lần chạy thử trước):
1. **Không bao giờ giải nén hàng loạt file nhỏ thẳng lên Drive** — copy zip về `/content/` (ổ đĩa cục bộ Colab) xử lý trước, chỉ ghi **kết quả gộp** (1 file duy nhất) lên Drive.
2. **Mọi bước tốn thời gian đều có checkpoint (`_manifest.txt`)** — nếu Colab ngắt giữa chừng, chạy lại notebook sẽ tự bỏ qua phần đã xong.
3. **Batch 1 đã có sẵn Keyframes/CLIP/Objects** — không cần tự trích xuất lại, chỉ cần đọc đúng và gộp thành index. Chỉ OCR, ASR, Audio embedding là phải tự làm từ đầu.

---

## Phần 0 — Setup: Mount Drive, cài thư viện, khai báo đường dẫn

**Giải thích:** đây là bước chuẩn bị môi trường — gắn nguồn dữ liệu (Drive trên Colab, hoặc Google Drive for Desktop trên máy cá nhân) để đọc/ghi dữ liệu, cài các thư viện cần dùng xuyên suốt notebook (FAISS cho vector search, EasyOCR cho đọc chữ, transformers cho ASR, rank_bm25 cho tìm kiếm văn bản), và tạo sẵn cấu trúc thư mục `extracted/` để lưu kết quả.

**Nếu chạy LOCAL (máy cá nhân):** cần cài sẵn Google Drive for Desktop, đăng nhập đúng tài khoản chứa dataset, và bật chế độ **Mirror** (hoặc đánh dấu **"Available offline"** riêng thư mục dataset) — xem lại phần đã bàn trước đó về ưu/nhược điểm 2 chế độ này. Sửa đúng đường dẫn ổ đĩa Drive trong cell khai báo path bên dưới trước khi chạy.


In [ ]:
import os
from pathlib import Path

# Phát hiện đang chạy ở đâu — Colab hay máy cá nhân (LOCAL)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
IS_LOCAL = not IS_COLAB

if IS_COLAB:
    print("Phát hiện đang chạy trên COLAB — mount Google Drive.")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Phát hiện đang chạy LOCAL (máy cá nhân) — dùng Google Drive for Desktop đã mount sẵn.")
    print("Đảm bảo Drive for Desktop đang chạy VÀ ở chế độ Mirror (hoặc offline riêng thư mục dataset).")


In [ ]:
!pip install faiss-cpu rank_bm25 pyarrow easyocr transformers librosa soundfile --quiet

In [ ]:
import json, zipfile, shutil, glob
from pathlib import Path
import numpy as np
import pandas as pd

SUBFOLDERS = ["clip_index", "objects", "media_info", "media_info_analysis", "map_keyframes",
              "ocr", "asr", "audio_embedding", "final_index"]

if IS_COLAB:
    # Sửa lại nếu cấu trúc thư mục Drive của bạn khác
    DATASET_ROOT = "/content/drive/MyDrive/AIC 2026/dataset"
    EXTRACTED_ROOT = os.path.join(DATASET_ROOT, "extracted")
    LOCAL_TMP = "/content/tmp"   # ổ đĩa cục bộ Colab, KHÔNG phải Drive — nhanh, mất khi session kết thúc
else:
    # LOCAL (máy cá nhân) — SỬA đúng đường dẫn Google Drive for Desktop trên máy bạn.
    # Xem đúng ký tự ổ đĩa trong File Explorer (Windows) — KHÔNG phải lúc nào cũng là G:
    DATASET_ROOT = r"G:\My Drive\AIC 2026\dataset"          # <-- SỬA CHỖ NÀY
    EXTRACTED_ROOT = os.path.join(DATASET_ROOT, "extracted")  # ghi thẳng vào Drive luôn — persist ngay như Colab
    LOCAL_TMP = os.path.join(os.environ.get("TEMP", "/tmp"), "aic2026_tmp")   # ổ cứng máy, không phải Drive

for sub in SUBFOLDERS:
    os.makedirs(os.path.join(EXTRACTED_ROOT, sub), exist_ok=True)
os.makedirs(LOCAL_TMP, exist_ok=True)

print("Nền tảng      :", "COLAB" if IS_COLAB else "LOCAL")
print("DATASET_ROOT  :", DATASET_ROOT)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("LOCAL_TMP     :", LOCAL_TMP)


### Hàm dùng chung — checkpoint theo manifest (áp dụng cho mọi bước dài phía sau)

In [ ]:
def load_manifest(manifest_path):
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            return set(f.read().splitlines())
    return set()

def mark_done(manifest_path, key):
    with open(manifest_path, "a") as f:
        f.write(key + "\n")

---
## Phần 1 — Bước 1: Ingest & Validate

**Giải thích:** kiểm tra tất cả file zip cần thiết đều tồn tại trước khi xử lý bất kỳ bước nào — tránh phát hiện thiếu file giữa chừng 1 batch job dài.

In [ ]:
checks = {
    "video/*.zip": list(Path(DATASET_ROOT, "video").glob("*.zip")),
    "keyframes/*.zip": list(Path(DATASET_ROOT, "keyframes").glob("*.zip")),
    "objects*.zip": list(Path(DATASET_ROOT).glob("objects*.zip")),
    "media-info*.zip": list(Path(DATASET_ROOT).glob("media-info*.zip")),
    "map-keyframes*.zip": list(Path(DATASET_ROOT).glob("map-keyframes*.zip")),
    "clip-features*.zip": list(Path(DATASET_ROOT).glob("clip-features*.zip")),
}

all_ok = True
for pattern, files in checks.items():
    status = "OK" if files else "THIẾU"
    if not files:
        all_ok = False
    print(f"[{status:5}] {pattern:22} — {len(files)} file")

print("\n=> Tất cả sẵn sàng" if all_ok else "\n=> CẢNH BÁO: thiếu 1 số file, kiểm tra lại DATASET_ROOT")

---
## Phần 2 — Bước 3a: CLIP Feature Extraction (đã có sẵn — gộp thành 1 FAISS Index)

**Giải thích:** file `clip-features*.zip` chứa các file `.npy` (1 file/video, shape `(N_keyframe, 512)`, `dtype=float16`, đã xác nhận thật ở `L30_V036.npy`). Việc cần làm: giải nén cục bộ (nhanh, vì file nhỏ), rồi **gộp toàn bộ vector của mọi video vào 1 ma trận lớn duy nhất**, build FAISS Index, và giữ lại bảng ánh xạ `(vị trí trong FAISS) → (video_id, chỉ số keyframe trong video)` để sau này biết vector nào thuộc video nào.

In [ ]:
# Kiểm tra đã có sẵn .npy giải nén trên Drive chưa (từ notebook Bước 0)
# -> nếu có, dùng THẲNG, KHÔNG tải/giải nén lại zip (tránh làm trùng việc)
existing_npy = sorted(Path(EXTRACTED_ROOT, "clip_features").rglob("*.npy"))

if existing_npy:
    print(f"Đã có sẵn {len(existing_npy)} file .npy trong extracted/clip_features/ — dùng thẳng.")
    npy_files = existing_npy
    clip_local_dir = None   # không tạo thư mục local -> không cần dọn dẹp sau này
else:
    print("Chưa có sẵn — giải nén clip-features từ zip về local...")
    clip_zip = str(next(Path(DATASET_ROOT).glob("clip-features*.zip")))
    clip_local_dir = os.path.join(LOCAL_TMP, "clip_features")
    os.makedirs(clip_local_dir, exist_ok=True)
    with zipfile.ZipFile(clip_zip, 'r') as zf:
        zf.extractall(clip_local_dir)
    npy_files = sorted(Path(clip_local_dir).rglob("*.npy"))

print(f"Số file .npy: {len(npy_files)}")
print(f"5 file đầu: {[f.name for f in npy_files[:5]]}")

In [ ]:
import faiss

all_vectors = []
mapping_records = []   # (video_id, local_frame_idx) tương ứng từng dòng trong all_vectors

for npy_path in npy_files:
    video_id = npy_path.stem   # VD: "L30_V036"
    arr = np.load(npy_path).astype("float32")   # ép float16 -> float32 (FAISS yêu cầu)
    all_vectors.append(arr)
    for local_idx in range(arr.shape[0]):
        mapping_records.append({"video_id": video_id, "local_frame_idx": local_idx})

clip_matrix = np.vstack(all_vectors)   # gộp tất cả video thành 1 ma trận (tổng_keyframe, 512)
print(f"Tổng số vector: {clip_matrix.shape[0]}, chiều: {clip_matrix.shape[1]}")

# Chuẩn hóa L2 trước khi build IndexFlatIP -> tương đương cosine similarity
faiss.normalize_L2(clip_matrix)

clip_index = faiss.IndexFlatIP(clip_matrix.shape[1])
clip_index.add(clip_matrix)
print(f"FAISS Index đã build xong: {clip_index.ntotal} vector")

In [ ]:
# Lưu FAISS Index + bảng mapping lên Drive — đây là kết quả DUY NHẤT cần giữ lại,
# không cần giữ lại toàn bộ .npy rời rạc nữa
faiss_index_path = os.path.join(EXTRACTED_ROOT, "clip_index", "clip_faiss.index")
mapping_path = os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet")

faiss.write_index(clip_index, faiss_index_path)
pd.DataFrame(mapping_records).to_parquet(mapping_path)

print(f"Đã lưu FAISS index : {faiss_index_path} ({os.path.getsize(faiss_index_path)/1e6:.2f} MB)")
print(f"Đã lưu mapping     : {mapping_path}")

# Chỉ dọn dẹp local nếu Phần 2 THỰC SỰ có tải/giải nén zip (clip_local_dir != None)
if clip_local_dir is not None:
    shutil.rmtree(clip_local_dir)

---
## Phần 3 — Bước 3b: Object Detection (đã có sẵn — gộp thành 1 Bảng)

**Giải thích:** đã xác nhận thật cấu trúc file JSON là định dạng TensorFlow Object Detection API (`detection_scores`, `detection_class_entities`, `detection_boxes`... — các mảng chạy SONG SONG, không phải danh sách object lồng nhau). Cell dưới đọc trực tiếp từng JSON trong zip (không giải nén ra đĩa), lọc theo ngưỡng tin cậy, gộp thành 1 bảng duy nhất.

In [ ]:
CONFIDENCE_THRESHOLD = 0.15
objects_path = os.path.join(EXTRACTED_ROOT, "objects", "objects_index.parquet")

# Nếu đã có sẵn bảng kết quả (từ lần chạy trước) -> bỏ qua toàn bộ xử lý,
# KHÔNG đọc lại từ extracted/objects/ (có thể chứa JSON dở dang từ lần giải nén
# thẳng lên Drive bị treo trước đó) — vẫn ưu tiên đọc từ zip nếu cần xử lý lại.
if os.path.exists(objects_path):
    print(f"Đã có sẵn {objects_path} — BỎ QUA xử lý lại.")
    print("Muốn làm lại từ đầu: xóa file này rồi chạy lại 3 cell của Phần 3.")
    SKIP_OBJECTS = True
else:
    SKIP_OBJECTS = False
    objects_zip = str(next(Path(DATASET_ROOT).glob("objects*.zip")))
    local_zip = os.path.join(LOCAL_TMP, "objects_temp.zip")
    shutil.copy(objects_zip, local_zip)
    print("Đã copy zip về local:", os.path.getsize(local_zip) / 1e6, "MB")

In [ ]:
if not SKIP_OBJECTS:
    records = []
    error_count = 0
    skipped_low_conf = 0

    with zipfile.ZipFile(local_zip, 'r') as zf:
        json_members = [n for n in zf.namelist() if n.endswith('.json')]
        total = len(json_members)
        print(f"Tổng số file JSON: {total}")
        print(f"Ví dụ đường dẫn nội bộ zip (kiểm tra để xác nhận video_id/frame_index tách đúng):")
        print(" ", json_members[:3])

        for i, name in enumerate(json_members):
            try:
                with zf.open(name) as f:
                    data = json.load(f)
            except Exception:
                error_count += 1
                continue

            path = Path(name)
            video_id = path.parent.name
            frame_index = path.stem

            scores   = data.get("detection_scores", [])
            entities = data.get("detection_class_entities", [])
            boxes    = data.get("detection_boxes", [])
            mids     = data.get("detection_class_names", [])

            for j in range(len(scores)):
                confidence = float(scores[j])
                if confidence < CONFIDENCE_THRESHOLD:
                    skipped_low_conf += 1
                    continue
                box = boxes[j] if j < len(boxes) else [None, None, None, None]
                records.append({
                    "video_id": video_id,
                    "frame_index": frame_index,
                    "label": entities[j] if j < len(entities) else None,
                    "confidence": confidence,
                    "bbox_ymin": float(box[0]), "bbox_xmin": float(box[1]),
                    "bbox_ymax": float(box[2]), "bbox_xmax": float(box[3]),
                    "class_mid": mids[j] if j < len(mids) else None,
                })

            if (i + 1) % 20000 == 0:
                print(f"  Đã xử lý {i+1}/{total} file...")

    print(f"\nHoàn tất. Số dòng giữ lại: {len(records)} | Bỏ qua (điểm thấp): {skipped_low_conf} | Lỗi: {error_count}")
else:
    print("Đã bỏ qua (SKIP_OBJECTS=True) — dùng file objects_index.parquet có sẵn.")

In [ ]:
if not SKIP_OBJECTS:
    objects_df = pd.DataFrame(records)
    objects_df.to_parquet(objects_path)
    print(f"Đã lưu: {objects_path} ({os.path.getsize(objects_path)/1e6:.2f} MB)")
    os.remove(local_zip)
else:
    objects_df = pd.read_parquet(objects_path)
    print(f"Đọc lại từ file có sẵn: {len(objects_df)} dòng")

print("\nTop 15 label phổ biến nhất:")
print(objects_df["label"].value_counts().head(15))

---
## Phần 6 — Bước 3b: Media-info Handling — ĐÃ CHUYỂN sang notebook riêng

**Cập nhật quan trọng:** bước này (đọc `media-info*.zip`, gộp thành bảng, sinh worklist) giờ được đảm nhiệm HOÀN TOÀN bởi notebook riêng **`AIC2026_Media_Info_Exploitation.ipynb`** — notebook đó làm đầy đủ hơn (còn phân tích thống kê + sinh worklist cho cả nhóm viết câu hỏi test), nên Offline notebook này **không tự làm lại** để tránh 2 nguồn cùng ghi đè lên `extracted/media_info_analysis/media_full.csv` với 2 schema khác nhau (từng xảy ra: bản cũ ở đây dùng cột `length`, bản kia dùng `length_seconds`).

Cell dưới đây chỉ KIỂM TRA xem `Media_Info_Exploitation.ipynb` đã chạy chưa (đủ 3 file chưa, đúng schema chưa) — nếu chưa, cảnh báo rõ ràng trước khi Phần 9.5 và notebook Online cần tới nó.


In [ ]:
media_analysis_dir = os.path.join(EXTRACTED_ROOT, "media_info_analysis")
required_files = ["media_full.csv", "worklist_kis_qa.csv", "worklist_trake.csv"]

print("Kiểm tra output của AIC2026_Media_Info_Exploitation.ipynb...")
all_present = True
for fname in required_files:
    fpath = os.path.join(media_analysis_dir, fname)
    if os.path.exists(fpath):
        print(f"  [OK]    {fname}")
    else:
        print(f"  [THIẾU] {fname}")
        all_present = False

if all_present:
    print("\nĐã có đủ 3 file — Phần 6 không cần làm gì thêm, dùng thẳng dữ liệu này ở các Phần sau.")
    _media_check_df = pd.read_csv(os.path.join(media_analysis_dir, "media_full.csv"), nrows=5)
    if "length_seconds" not in _media_check_df.columns:
        print("[CẢNH BÁO] Không thấy cột 'length_seconds' trong media_full.csv — Online")
        print("clip_frame_number() cần ĐÚNG tên cột này để giới hạn frame TRAKE chính xác.")
        print("Kiểm tra lại schema của Media_Info_Exploitation.ipynb.")
    else:
        print("Cột 'length_seconds' có mặt — khớp đúng với Online clip_frame_number().")
else:
    print("\n[CẢNH BÁO] Thiếu file — hãy chạy AIC2026_Media_Info_Exploitation.ipynb trước khi tiếp")
    print("tục, nếu không Phần 9.5 (kiểm tra chéo video_id) và notebook Online sẽ báo thiếu Media-info.")


---
## Phần 5 — Map-keyframes: mapping filename keyframe → frame_id thật

**Giải thích:** đây là bảng QUAN TRỌNG NHẤT về mặt rủi ro — sai ở đây khiến toàn bộ kết quả nộp bài sai tuyệt đối dù nội dung tìm đúng (đã bàn kỹ ở Bước 1). Vì chưa xác nhận định dạng thật của file này (CSV hay JSON), cell đầu tiên chỉ IN RA nội dung thô để xác nhận trước khi parse chính thức.

In [ ]:
map_path = os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet")

# Kiểm tra ĐÃ XONG chưa — LẦN NÀY xác nhận file THỰC SỰ CÓ DỮ LIỆU, không chỉ
# kiểm tra "tồn tại" (bài học từ chính lỗi vừa gặp: file có thể tồn tại nhưng
# RỖNG, do bị ghi đè bởi 1 lần chạy lỗi trước đó).
SKIP_MK = False
if os.path.exists(map_path):
    try:
        _check_df = pd.read_parquet(map_path)
        if len(_check_df) > 0 and "video_id" in _check_df.columns:
            print(f"Đã có sẵn kết quả HỢP LỆ: {map_path} ({len(_check_df)} dòng) — BỎ QUA dò/parse lại.")
            SKIP_MK = True
        else:
            print(f"[CẢNH BÁO] File tồn tại nhưng RỖNG/không hợp lệ "
                  f"({len(_check_df)} dòng, cột: {list(_check_df.columns)}) -> sẽ TẠO LẠI từ đầu.")
    except Exception as e:
        print(f"[CẢNH BÁO] File tồn tại nhưng đọc bị lỗi ({e}) -> sẽ TẠO LẠI từ đầu.")

if not SKIP_MK:
    # Chỉ liệt kê file NGUỒN thật (loại trừ CẢ _manifest.txt LẪN chính file .parquet đầu ra —
    # dù giờ nó rỗng, vẫn không được coi là "nguồn" để tránh lặp lại đúng lỗi cũ)
    existing_mk_files = [p for p in sorted(Path(EXTRACTED_ROOT, "map_keyframes").glob("*"))
                          if p.is_file() and p.name not in ("_manifest.txt", "map_keyframes_index.parquet")]

    if existing_mk_files:
        print(f"\nĐã có sẵn {len(existing_mk_files)} file NGUỒN trong extracted/map_keyframes/ — dùng thẳng, bỏ qua zip.")
        USE_EXISTING_MK = True
        sample_path = existing_mk_files[0]
        print(f"\n--- Nội dung thô của {sample_path.name} (2000 ký tự đầu) ---")
        with open(sample_path, "r", errors="ignore") as f:
            print(f.read(2000))
    else:
        USE_EXISTING_MK = False
        mk_zip = str(next(Path(DATASET_ROOT).glob("map-keyframes*.zip")))
        with zipfile.ZipFile(mk_zip, 'r') as zf:
            members = zf.namelist()
            print(f"\nKhông có file nguồn rời rạc -> đọc từ zip. Tổng số file: {len(members)}")
            print("5 file đầu:", members[:5])

            sample_name = members[0]
            with zf.open(sample_name) as f:
                raw = f.read(2000)
            print(f"\n--- Nội dung thô của {sample_name} (2000 byte đầu) ---")
            print(raw.decode("utf-8", errors="ignore"))

**Dừng lại đọc kỹ output ở trên trước khi chạy cell tiếp theo.** Cell dưới đây giả định định dạng CSV với cột `n` (thứ tự keyframe) và `frame_idx` (frame_id thật trong video) — đây là format phổ biến ở các bộ dữ liệu AIC các năm trước. **Nếu output ở trên cho thấy định dạng khác (VD: JSON, hoặc tên cột khác), cần sửa lại cell bên dưới cho khớp** trước khi chạy tiếp.

In [ ]:
import io
from collections import Counter

if not SKIP_MK:
    map_records = []
    error_count = 0
    skipped_suffix_count = Counter()   # đếm suffix bị bỏ qua — KHÔNG còn im lặng như bản trước

    if USE_EXISTING_MK:
        for path in existing_mk_files:
            video_id = path.stem
            suffix = path.suffix.lower()
            try:
                if suffix == ".csv":
                    df_tmp = pd.read_csv(path)
                elif suffix == ".json":
                    with open(path, encoding="utf-8") as f:
                        data = json.load(f)
                    df_tmp = pd.DataFrame(data)
                elif suffix in (".txt", ".tsv"):
                    df_tmp = pd.read_csv(path, sep=None, engine="python")
                else:
                    skipped_suffix_count[suffix or "(không đuôi)"] += 1
                    continue
                df_tmp["video_id"] = video_id
                map_records.append(df_tmp)
            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(f"  [LỖI] {path.name}: {e}")
    else:
        with zipfile.ZipFile(mk_zip, 'r') as zf:
            members = zf.namelist()
            print(f"Tổng số file trong zip: {len(members)}")
            for name in members:
                video_id = Path(name).stem
                suffix = Path(name).suffix.lower()
                try:
                    with zf.open(name) as f:
                        if suffix == ".csv":
                            df_tmp = pd.read_csv(io.TextIOWrapper(f, encoding='utf-8'))
                        elif suffix == ".json":
                            data = json.load(f)
                            df_tmp = pd.DataFrame(data)
                        elif suffix in (".txt", ".tsv"):
                            df_tmp = pd.read_csv(io.TextIOWrapper(f, encoding='utf-8'), sep=None, engine="python")
                        else:
                            skipped_suffix_count[suffix or "(không đuôi)"] += 1
                            continue
                    df_tmp["video_id"] = video_id
                    map_records.append(df_tmp)
                except Exception as e:
                    error_count += 1
                    if error_count <= 3:
                        print(f"  [LỖI] {name}: {e}")

    map_df = pd.concat(map_records, ignore_index=True) if map_records else pd.DataFrame()
    print(f"\nĐã đọc: {len(map_df)} dòng | File lỗi: {error_count}")
    if skipped_suffix_count:
        print(f"[CẢNH BÁO] Bỏ qua {sum(skipped_suffix_count.values())} file vì KHÔNG khớp .csv/.json/.txt/.tsv:")
        for suf, cnt in skipped_suffix_count.most_common():
            print(f"    suffix='{suf}' -> {cnt} file")
else:
    map_df = pd.read_parquet(map_path)
    print(f"Đọc lại từ file có sẵn: {len(map_df)} dòng")

print("\nCác cột thực tế đọc được:", list(map_df.columns))
print(map_df.head())

In [ ]:
if not SKIP_MK:
    map_df.to_parquet(map_path)
    print(f"Đã lưu: {map_path}")
else:
    print(f"Đã dùng file có sẵn, không cần lưu lại: {map_path}")

---
## Phần 6 — Bước 3c: OCR (tự làm — chưa có sẵn)

**Giải thích:** đây là bước NẶNG nhất trong notebook — phải chạy model OCR trên TOÀN BỘ keyframe. Chiến lược: xử lý **từng zip keyframes một** (streaming) — giải nén cục bộ 1 zip, chạy OCR, lưu kết quả (chỉ text, rất nhẹ) lên Drive, XÓA ảnh cục bộ, rồi mới sang zip tiếp theo. Có checkpoint theo TỪNG ZIP (không phải từng ảnh) để cân bằng giữa an toàn và overhead ghi manifest.

**Vì sao KHÔNG đổi sang đọc trực tiếp từ `extracted/keyframes/` dù đã có sẵn:** khác với CLIP/Objects/Media-info/Map-keyframes (ít file, nhẹ), `keyframes/` chứa **hàng trăm nghìn ảnh rời rạc**. Dù đã giải nén sẵn trên Drive, mở TỪNG ảnh riêng lẻ từ Drive vẫn chậm y hệt vấn đề đã gặp ở `objects/` (mỗi lần mở 1 file = 1 lượt gọi mạng). Đọc tuần tự từ bên trong 1 file zip (nén liên tục, đọc cục bộ) vẫn nhanh hơn nhiều so với mở hàng loạt file rời đã giải nén trên Drive — nên notebook này **cố ý giữ nguyên** cách đọc từ zip cho riêng bước này.

**Cảnh báo:** bước này có thể mất RẤT NHIỀU giờ tùy tổng số keyframe. Nên chạy thử trên 1 zip nhỏ trước (cell tiếp theo có tham số `LIMIT_ZIPS` để giới hạn số zip xử lý khi test).

In [ ]:
import easyocr
ocr_reader = easyocr.Reader(['vi', 'en'], gpu=True)
print("Đã load EasyOCR (tiếng Việt + tiếng Anh)")

In [ ]:
LIMIT_ZIPS = None   # đặt = 1 hoặc 2 để TEST trước; đặt = None để chạy TOÀN BỘ

ocr_dest = os.path.join(EXTRACTED_ROOT, "ocr")
ocr_manifest = os.path.join(ocr_dest, "_manifest.txt")
done_zips = load_manifest(ocr_manifest)

keyframe_zips = sorted(Path(DATASET_ROOT, "keyframes").glob("*.zip"))
if LIMIT_ZIPS:
    keyframe_zips = keyframe_zips[:LIMIT_ZIPS]

print(f"Sẽ xử lý {len(keyframe_zips)} zip keyframes (đã xong trước đó: {len(done_zips)})")

In [ ]:
for zip_path in keyframe_zips:
    zip_name = zip_path.name
    if zip_name in done_zips:
        print(f"[SKIP] {zip_name} đã xử lý trước đó")
        continue

    print(f"[BẮT ĐẦU] {zip_name}")
    local_dir = os.path.join(LOCAL_TMP, "kf_temp")
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    os.makedirs(local_dir)

    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        zf.extractall(local_dir)

    image_files = sorted(Path(local_dir).rglob("*.jpg")) + sorted(Path(local_dir).rglob("*.png"))
    print(f"  Số ảnh trong zip này: {len(image_files)}")

    zip_records = []
    for i, img_path in enumerate(image_files):
        video_id = img_path.parent.name
        frame_index = img_path.stem
        try:
            results = ocr_reader.readtext(str(img_path), detail=0)   # detail=0: chỉ lấy text, bỏ tọa độ
            text = " ".join(results).strip()
        except Exception:
            text = ""
        if text:   # chỉ lưu nếu THỰC SỰ đọc được chữ, tránh phình bảng với dòng rỗng
            zip_records.append({"video_id": video_id, "frame_index": frame_index, "ocr_text": text})

        if (i + 1) % 500 == 0:
            print(f"    ...{i+1}/{len(image_files)} ảnh")

    # Ghi kết quả của ZIP NÀY vào 1 file riêng trên Drive (tránh phải đọc/ghi lại
    # toàn bộ file lớn mỗi zip — mỗi zip 1 file nhỏ, gộp lại ở Phần 8)
    out_path = os.path.join(ocr_dest, f"ocr_{zip_path.stem}.parquet")
    pd.DataFrame(zip_records).to_parquet(out_path)
    mark_done(ocr_manifest, zip_name)
    print(f"[XONG] {zip_name} -> {len(zip_records)} dòng có text, lưu tại {out_path}")

    shutil.rmtree(local_dir)   # dọn local ngay, tránh đầy disk Colab

---
## Phần 7 — Bước 4a + 4b: Audio Extraction + ASR (tự làm — chưa có sẵn)

**Giải thích:** tương tự OCR — xử lý từng zip `video/` một. Với mỗi video: tách audio bằng ffmpeg, chạy PhoWhisper để có transcript, lưu transcript (nhẹ) lên Drive, xóa video + audio cục bộ ngay sau đó (video rất nặng, không thể giữ lại).

**Checkpoint theo TỪNG VIDEO** (không phải theo zip) — mỗi video được lưu ra 1 file `asr_<video_id>.parquet` + đánh dấu vào `_manifest.txt` NGAY sau khi xử lý xong, không đợi hết cả zip. Nếu Colab ngắt/hết giờ giữa chừng 1 zip, các video đã xong KHÔNG bị mất — chạy lại notebook sẽ tự động bỏ qua chúng và chỉ tiếp tục đúng phần dang dở.

**Vì sao KHÔNG đổi sang đọc trực tiếp từ `extracted/video/` dù đã có sẵn:** cùng lý do như OCR — file video rất nặng, và dù đã giải nén sẵn, đọc/copy từng file .mp4 riêng lẻ từ Drive không nhanh hơn đáng kể so với đọc từ trong zip (Drive I/O là nút thắt chính, không phải bước giải nén). Notebook này giữ nguyên cách đọc từ zip cho bước này.

In [ ]:
from transformers import pipeline as hf_pipeline

# FIX BUG NGHIÊM TRỌNG: thiếu chunk_length_s khiến Whisper tự CẮT CỤT mọi audio về đúng 30
# giây đầu tiên (giới hạn input gốc của kiến trúc Whisper) — phần còn lại của video bị bỏ
# hoàn toàn, KHÔNG báo lỗi gì. Đây là nguyên nhân toàn bộ 873/873 video chỉ ra rất ít dòng
# ASR (trung bình 2.7 dòng/video, cao nhất chỉ 10 dòng) dù nhiều video dài ~20 phút.
# chunk_length_s=30 bật đúng cơ chế chia audio dài thành nhiều đoạn 30s xử lý tuần tự;
# stride_length_s=5 cho 2 đoạn liền kề chồng lên nhau 5s để không cắt đứt giữa câu.
asr_pipe = hf_pipeline("automatic-speech-recognition", model="vinai/PhoWhisper-base", device=0,
                       chunk_length_s=30, stride_length_s=5)
print("Đã load PhoWhisper-base (chunk_length_s=30 -> xử lý được audio dài, không còn cắt cụt 30s đầu)")


In [ ]:
LIMIT_VIDEO_ZIPS = None   # đặt = 1 để TEST trước; None để chạy toàn bộ

asr_dest = os.path.join(EXTRACTED_ROOT, "asr")
asr_manifest = os.path.join(asr_dest, "_manifest.txt")
# FIX: checkpoint giờ theo TỪNG VIDEO, không phải theo zip — trước đây nếu bị ngắt giữa
# chừng 1 zip (VD đang xử lý video thứ 15/29), toàn bộ 14 video đã xong bị MẤT (chỉ nằm
# trong RAM, chưa ghi ra đĩa), lần chạy lại phải làm lại từ đầu CẢ ZIP. Giờ mỗi video
# được lưu + đánh dấu NGAY sau khi xong, không đợi hết cả zip mới ghi.
done_asr_videos = load_manifest(asr_manifest)

video_zips = sorted(Path(DATASET_ROOT, "video").glob("*.zip"))
if LIMIT_VIDEO_ZIPS:
    video_zips = video_zips[:LIMIT_VIDEO_ZIPS]

print(f"Sẽ xử lý {len(video_zips)} zip video (đã xong trước đó: {len(done_asr_videos)} video)")


### 7.0a — XÓA dữ liệu ASR cũ (do bug thiếu `chunk_length_s`, chạy 1 LẦN DUY NHẤT)

**Giải thích:** vì bug thiếu `chunk_length_s` ảnh hưởng **toàn bộ 873/873 video** (không phải 1 phần), và checkpoint đã đánh dấu tất cả là "đã xong" — nếu không xóa, pipeline (đã fix) sẽ `[SKIP]` hết vì tưởng nhầm đã có kết quả, trong khi kết quả đó chỉ là 30 giây đầu mỗi video. Cell này xóa sạch toàn bộ `asr_*.parquet` + reset `_manifest.txt`, để chạy lại từ đầu với pipeline đã sửa.


In [ ]:
# Đổi thành True, chạy Ô NÀY 1 LẦN DUY NHẤT để xóa dữ liệu ASR cũ (bị lỗi do thiếu
# chunk_length_s), sau đó đổi lại False để tránh vô tình xóa lần sau.
RESET_ASR_DATA = False

if RESET_ASR_DATA:
    old_parts = list(Path(asr_dest).glob("asr_*.parquet"))
    for p in old_parts:
        p.unlink()
    if os.path.exists(asr_manifest):
        os.remove(asr_manifest)
    done_asr_videos = set()
    print(f"Đã xóa {len(old_parts)} file ASR cũ + reset manifest.")
    print("Sẵn sàng chạy lại từ đầu với pipeline đã fix (chunk_length_s=30).")
else:
    print("RESET_ASR_DATA=False — chưa xóa gì.")
    print("Đổi thành True rồi chạy lại Ô NÀY nếu muốn xóa ASR cũ (do bug chunk_length_s).")


### 7.0 — Dọn file checkpoint ĐỊNH DẠNG CŨ (chạy 1 lần, an toàn để chạy lại nhiều lần)

**Giải thích:** bản code cũ (trước khi checkpoint được sửa theo từng video) lưu file dạng `asr_<tên_zip>.parquet` (VD `asr_Videos_L21_a.parquet`, gộp NHIỀU video trong 1 file). Bản mới lưu `asr_<video_id>.parquet` (1 video/file) — 2 định dạng không tương thích, khiến `_manifest.txt` cũ (chứa tên zip) không khớp được với video_id, làm code hiểu nhầm là CHƯA xử lý và chạy lại từ đầu. Cell này tách các file cũ thành đúng định dạng mới (KHÔNG mất dữ liệu — transcript đã có sẵn trong file cũ được giữ nguyên, chỉ tách lại theo video), xóa file cũ sau khi tách xong, và cập nhật `done_asr_videos` ngay trong session hiện tại.


In [ ]:
old_format_files = list(Path(asr_dest).glob("asr_Videos_*.parquet"))
print(f"Tìm thấy {len(old_format_files)} file ASR định dạng CŨ (theo zip):")
for p in old_format_files:
    print(" -", p.name)

if old_format_files:
    for old_path in old_format_files:
        df = pd.read_parquet(old_path)
        if "video_id" not in df.columns or len(df) == 0:
            print(f"  [BỎ QUA] {old_path.name}: không có cột video_id hoặc rỗng")
            old_path.unlink()
            continue

        n_videos = df["video_id"].nunique()
        for vid, group in df.groupby("video_id"):
            new_path = os.path.join(asr_dest, f"asr_{vid}.parquet")
            if not os.path.exists(new_path):   # không ghi đè nếu lỡ đã có bản mới hơn
                group.to_parquet(new_path)
            if vid not in done_asr_videos:
                done_asr_videos.add(vid)
                mark_done(asr_manifest, vid)

        old_path.unlink()   # xóa file cũ SAU KHI tách xong — tránh Phần 9 đếm trùng
        print(f"  [XONG] {old_path.name} -> tách thành {n_videos} file theo video, đã xóa file cũ")

    print(f"\ndone_asr_videos giờ có {len(done_asr_videos)} video hợp lệ (đã cập nhật ngay trong session này).")
else:
    print("Không có file định dạng cũ nào cần dọn — checkpoint đã sạch, chạy tiếp bình thường.")


In [ ]:
for zip_path in video_zips:
    zip_name = zip_path.name

    # Đọc tên video TRONG zip trước (không giải nén) để biết có thể skip sớm hay không
    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        video_ids_in_zip = [Path(n).stem for n in zf.namelist() if n.lower().endswith(".mp4")]

    if video_ids_in_zip and all(vid in done_asr_videos for vid in video_ids_in_zip):
        print(f"[SKIP] {zip_name} — tất cả {len(video_ids_in_zip)} video đã xử lý xong trước đó")
        continue

    print(f"[BẮT ĐẦU] {zip_name}")
    local_dir = os.path.join(LOCAL_TMP, "video_temp")
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    os.makedirs(local_dir)

    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        zf.extractall(local_dir)

    video_files = sorted(Path(local_dir).rglob("*.mp4"))
    videos_to_process = [v for v in video_files if v.stem not in done_asr_videos]
    print(f"  Số video trong zip này: {len(video_files)} "
          f"(cần xử lý: {len(videos_to_process)}, đã xong trước đó: {len(video_files) - len(videos_to_process)})")

    for v in videos_to_process:
        video_id = v.stem
        audio_path = os.path.join(LOCAL_TMP, f"{video_id}.wav")
        ffmpeg_ret = os.system(f'ffmpeg -y -i "{v}" -vn -acodec pcm_s16le -ar 16000 "{audio_path}" -loglevel quiet')

        if ffmpeg_ret != 0 or not os.path.exists(audio_path) or os.path.getsize(audio_path) == 0:
            print(f"    [LỖI ffmpeg] {video_id}: extract audio thất bại (return code {ffmpeg_ret}) -> bỏ qua")
            if os.path.exists(audio_path):
                os.remove(audio_path)
            # KHÔNG mark_done — để lần chạy sau tự động thử lại đúng video này,
            # không bị coi nhầm là "đã xong" trong khi thực ra chưa có kết quả gì.
            continue

        video_records = []
        try:
            result = asr_pipe(audio_path, return_timestamps=True)
            for chunk in result.get("chunks", []):
                video_records.append({
                    "video_id": video_id,
                    "start": chunk["timestamp"][0],
                    "end": chunk["timestamp"][1],
                    "asr_text": chunk["text"].strip(),
                })
        except Exception as e:
            print(f"    [LỖI ASR] {video_id}: {e}")

        if os.path.exists(audio_path):
            os.remove(audio_path)

        # FIX: lưu + đánh dấu NGAY sau khi xong 1 video (kể cả khi video_records rỗng do
        # lỗi ASR — vẫn mark_done để không lặp lại vô ích 1 lỗi nhiều khả năng sẽ lặp lại
        # y hệt ở lần chạy sau). Đây chính là điểm khác biệt cốt lõi so với bản cũ: ghi
        # NGAY, không đợi cộng dồn hết cả zip mới ghi 1 lần.
        out_path = os.path.join(asr_dest, f"asr_{video_id}.parquet")
        pd.DataFrame(video_records).to_parquet(out_path)
        mark_done(asr_manifest, video_id)

    print(f"[XONG] {zip_name}")
    shutil.rmtree(local_dir)


---
## Phần 8 — Bước 4c: Audio Embedding (CLAP — đã triển khai đầy đủ)

**Giải thích:** dùng model **CLAP** (qua `transformers.ClapModel` — KHÔNG dùng gói PyPI `laion-clap` riêng, xem lý do ở cell dưới) — cùng nguyên lý với CLIP nhưng cho cặp audio-text thay vì ảnh-text, cùng chung 1 không gian vector để so khớp text query với đoạn âm thanh. Khác với CLIP (encode theo TỪNG keyframe tĩnh), audio mang tính LIÊN TỤC theo thời gian nên chia theo **cửa sổ thời gian cố định** (`AUDIO_WINDOW_SECONDS`, mặc định 5 giây), không theo keyframe — giống hệt cách ASR (Phần 7) đã lưu `start`/`end` theo giây, quy đổi sang `frame_id` ở thời điểm truy vấn (Online) bằng `asr_timestamp_to_frame_id()` đã có sẵn.

**Chiến lược xử lý** (giống hệt Phần 7 — ASR): xử lý từng zip `video/` một, tách audio bằng ffmpeg, cắt thành các đoạn `AUDIO_WINDOW_SECONDS` giây, encode bằng CLAP, lưu **vector + metadata riêng theo từng zip** lên Drive (checkpoint theo `_manifest.txt`), xóa audio cục bộ ngay sau đó. Sau khi xử lý xong toàn bộ zip, 1 cell riêng sẽ **gộp lại** thành 1 FAISS Index + 1 bảng mapping duy nhất (giống cách Phần 9 gộp OCR+ASR thành BM25 Index).


In [ ]:
# FIX: gói PyPI "laion-clap" ép hạ numpy xuống 1.26.4 -> xung đột nặng với hàng loạt
# thư viện khác trong Colab cần numpy>=2.0 (jax, opencv, tensorflow, cupy...), kéo theo
# downgrade cả triton, làm gãy torch._inductor mà CLIP ở Phần 2 đang dùng (lỗi
# "AttributeError: module 'triton.backends' has no attribute 'compiler'"). Dùng thẳng
# CLAP qua transformers.ClapModel (đã hỗ trợ native, tương thích numpy 2.x) thay vì gói
# laion-clap riêng -> tránh HOÀN TOÀN xung đột dependency này.
!pip install -U transformers --quiet


In [ ]:
import torch
from transformers import ClapModel, ClapProcessor
import numpy as np
import librosa

CLAP_CHECKPOINT = "laion/clap-htsat-unfused"
clap_device = "cuda" if torch.cuda.is_available() else "cpu"

clap_model = ClapModel.from_pretrained(CLAP_CHECKPOINT).to(clap_device)
clap_processor = ClapProcessor.from_pretrained(CLAP_CHECKPOINT)
clap_model.eval()
print(f"Đã load CLAP model ({CLAP_CHECKPOINT}) qua transformers, chạy trên: {clap_device}")
print("Encode CẢ audio lẫn text vào chung 1 không gian vector (giống nguyên lý CLIP)")

def _extract_clap_features(output):
    """transformers bản mới đang chuẩn hoá lại get_audio_features()/get_text_features()
    -> có thể trả về BaseModelOutputWithPooling thay vì tensor thuần như trước (xem GitHub
    issue huggingface/transformers#42401). Lấy pooler_output nếu có, fallback về chính
    object nếu không (coi như tensor thuần — tương thích ngược với bản cũ)."""
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    return output



In [ ]:
# Độ dài mỗi đoạn audio được encode — CLAP hoạt động tốt với clip ngắn (~5-10s). KHÔNG chia
# theo từng keyframe (khác CLIP ảnh) vì âm thanh mang tính liên tục theo thời gian, không phải
# khoảnh khắc tĩnh — quy đổi sang frame_id thật sẽ làm ở Online lúc truy vấn, giống cơ chế ASR.
AUDIO_WINDOW_SECONDS = 5.0
CLAP_SAMPLE_RATE = 48000   # đúng sample rate CLAP được train — tránh resample sai làm giảm chất lượng

audio_emb_dest = os.path.join(EXTRACTED_ROOT, "audio_embedding")
audio_manifest = os.path.join(audio_emb_dest, "_manifest.txt")
done_audio_zips = load_manifest(audio_manifest)

video_zips_for_audio = sorted(Path(DATASET_ROOT, "video").glob("*.zip"))
LIMIT_AUDIO_ZIPS = None   # đặt = 1 để TEST trước; None để chạy toàn bộ
if LIMIT_AUDIO_ZIPS:
    video_zips_for_audio = video_zips_for_audio[:LIMIT_AUDIO_ZIPS]

print(f"Sẽ xử lý {len(video_zips_for_audio)} zip video cho Audio Embedding "
      f"(đã xong trước đó: {len(done_audio_zips)})")


In [ ]:
for zip_path in video_zips_for_audio:
    zip_name = zip_path.name
    if zip_name in done_audio_zips:
        print(f"[SKIP] {zip_name} đã xử lý trước đó")
        continue

    print(f"[BẮT ĐẦU] {zip_name}")
    local_dir = os.path.join(LOCAL_TMP, "audio_video_temp")
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    os.makedirs(local_dir)

    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        zf.extractall(local_dir)

    video_files = sorted(Path(local_dir).rglob("*.mp4"))
    print(f"  Số video trong zip này: {len(video_files)}")

    zip_records = []
    zip_vectors = []

    for v in video_files:
        video_id = v.stem
        audio_path = os.path.join(LOCAL_TMP, f"{video_id}_clap.wav")
        ffmpeg_ret = os.system(
            f'ffmpeg -y -i "{v}" -vn -ac 1 -ar {CLAP_SAMPLE_RATE} "{audio_path}" -loglevel quiet')

        # Check ffmpeg thành công trước khi xử lý tiếp (cùng nguyên tắc đã áp dụng cho ASR ở Phần 7)
        if ffmpeg_ret != 0 or not os.path.exists(audio_path) or os.path.getsize(audio_path) == 0:
            print(f"    [LỖI ffmpeg] {video_id}: extract audio thất bại -> bỏ qua")
            if os.path.exists(audio_path):
                os.remove(audio_path)
            continue

        try:
            waveform, sr = librosa.load(audio_path, sr=CLAP_SAMPLE_RATE, mono=True)
        except Exception as e:
            print(f"    [LỖI librosa] {video_id}: {e}")
            os.remove(audio_path)
            continue

        window_samples = int(AUDIO_WINDOW_SECONDS * CLAP_SAMPLE_RATE)
        n_windows = max(1, int(np.ceil(len(waveform) / window_samples)))

        video_chunks = []
        video_windows_meta = []
        for w in range(n_windows):
            start_sample = w * window_samples
            end_sample = min(start_sample + window_samples, len(waveform))
            chunk = waveform[start_sample:end_sample]
            if len(chunk) < CLAP_SAMPLE_RATE * 0.5:   # bỏ đoạn cuối quá ngắn (<0.5s), không đủ tín hiệu
                continue
            if len(chunk) < window_samples:   # CLAP cần đúng độ dài cố định -> pad 0 nếu đoạn cuối ngắn hơn
                chunk = np.pad(chunk, (0, window_samples - len(chunk)))
            video_chunks.append(chunk)
            video_windows_meta.append({
                "video_id": video_id,
                "start_sec": start_sample / CLAP_SAMPLE_RATE,
                "end_sec": end_sample / CLAP_SAMPLE_RATE,
            })

        if video_chunks:
            clap_inputs = clap_processor(audio=video_chunks, sampling_rate=CLAP_SAMPLE_RATE,
                                          return_tensors="pt")
            clap_inputs = {k: v.to(clap_device) for k, v in clap_inputs.items()}
            with torch.no_grad():
                audio_out = clap_model.get_audio_features(**clap_inputs)
                embeddings = _extract_clap_features(audio_out).cpu().numpy().astype("float32")
            zip_vectors.append(embeddings)
            zip_records.extend(video_windows_meta)

        os.remove(audio_path)

    if zip_vectors:
        zip_matrix = np.vstack(zip_vectors).astype("float32")
        out_vec_path = os.path.join(audio_emb_dest, f"audio_vectors_{zip_path.stem}.npy")
        out_meta_path = os.path.join(audio_emb_dest, f"audio_meta_{zip_path.stem}.parquet")
        np.save(out_vec_path, zip_matrix)
        pd.DataFrame(zip_records).to_parquet(out_meta_path)
        print(f"[XONG] {zip_name} -> {len(zip_records)} đoạn audio ({zip_matrix.shape}), "
              f"lưu tại {out_vec_path}")
    else:
        print(f"[XONG] {zip_name} -> 0 đoạn audio hợp lệ")

    mark_done(audio_manifest, zip_name)
    shutil.rmtree(local_dir)


---
### 8b — Gộp toàn bộ part nhỏ lẻ thành 1 FAISS Index + 1 bảng mapping

**Giải thích:** giống hệt cách Phần 9 gộp các file OCR/ASR nhỏ lẻ thành 1 BM25 Index — mỗi zip ở trên tạo 2 file part riêng (`audio_vectors_*.npy` + `audio_meta_*.parquet`), cell này đọc TOÀN BỘ part, gộp thành 1 ma trận vector duy nhất, build FAISS Index, và lưu bảng mapping `(video_id, start_sec, end_sec)` — thứ tự dòng khớp CHÍNH XÁC với vị trí vector trong FAISS (đúng quy ước đã dùng cho `clip_mapping.parquet` ở Phần 2).


In [ ]:
audio_vector_parts = sorted(Path(audio_emb_dest).glob("audio_vectors_*.npy"))
audio_meta_parts = sorted(Path(audio_emb_dest).glob("audio_meta_*.parquet"))

if not audio_vector_parts:
    print("Chưa có phần audio embedding nào — chạy lại vòng lặp phía trên trước.")
else:
    all_vectors = [np.load(p) for p in audio_vector_parts]
    all_meta = [pd.read_parquet(p) for p in audio_meta_parts]

    audio_matrix = np.vstack(all_vectors).astype("float32")
    audio_mapping_df = pd.concat(all_meta, ignore_index=True)
    assert len(audio_mapping_df) == audio_matrix.shape[0], "Số dòng metadata không khớp số vector!"

    faiss.normalize_L2(audio_matrix)   # chuẩn hóa L2 -> IndexFlatIP tương đương cosine similarity
    audio_faiss_index = faiss.IndexFlatIP(audio_matrix.shape[1])
    audio_faiss_index.add(audio_matrix)

    faiss_path = os.path.join(audio_emb_dest, "audio_faiss.index")
    mapping_path = os.path.join(audio_emb_dest, "audio_mapping.parquet")
    faiss.write_index(audio_faiss_index, faiss_path)
    audio_mapping_df.to_parquet(mapping_path)

    print(f"Audio FAISS Index: {audio_faiss_index.ntotal} vector, {audio_matrix.shape[1]} chiều")
    print(f"Đã lưu: {faiss_path}")
    print(f"Đã lưu: {mapping_path}")

    # Dọn các file part nhỏ lẻ SAU KHI đã gộp + lưu thành công (giữ lại _manifest.txt để resume đúng)
    for p in audio_vector_parts + audio_meta_parts:
        os.remove(p)
    print(f"Đã dọn {len(audio_vector_parts) + len(audio_meta_parts)} file part nhỏ lẻ.")


---
## Phần 9 — Bước 5: Build Index tổng hợp (gộp OCR + ASR thành 1 BM25 Index)

**Giải thích:** Phần 6 và 7 đã lưu kết quả OCR/ASR thành **nhiều file nhỏ** (1 file/zip, để an toàn khi checkpoint). Bước này gộp toàn bộ các file nhỏ đó lại thành **1 bảng văn bản duy nhất**, rồi build BM25 Index để Text Search (Online) tra cứu.

In [ ]:
from rank_bm25 import BM25Okapi

# Gộp toàn bộ file OCR nhỏ lẻ
ocr_parts = [pd.read_parquet(p) for p in Path(EXTRACTED_ROOT, "ocr").glob("ocr_*.parquet")]
ocr_all = pd.concat(ocr_parts, ignore_index=True) if ocr_parts else pd.DataFrame(columns=["video_id","frame_index","ocr_text"])
ocr_all["source"] = "OCR"
ocr_all = ocr_all.rename(columns={"ocr_text": "text"})

# Gộp toàn bộ file ASR nhỏ lẻ
asr_parts = [pd.read_parquet(p) for p in Path(EXTRACTED_ROOT, "asr").glob("asr_*.parquet")]
asr_all = pd.concat(asr_parts, ignore_index=True) if asr_parts else pd.DataFrame(columns=["video_id","start","end","asr_text"])
asr_all["source"] = "ASR"
asr_all = asr_all.rename(columns={"asr_text": "text"})

# Gộp chung OCR + ASR thành 1 bảng text duy nhất (đúng thiết kế Bước 5 trong Notion)
text_index_df = pd.concat([ocr_all, asr_all], ignore_index=True)
print(f"Tổng số dòng text (OCR + ASR): {len(text_index_df)}")
print(text_index_df["source"].value_counts())

In [ ]:
import pickle

# Build BM25 — cần tokenize đơn giản (tách theo khoảng trắng, đủ dùng cho bước này;
# có thể nâng cấp bằng thư viện tách từ tiếng Việt chuyên dụng sau)
tokenized_corpus = [str(t).lower().split() for t in text_index_df["text"].fillna("")]
bm25 = BM25Okapi(tokenized_corpus)

text_index_path = os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet")
bm25_path = os.path.join(EXTRACTED_ROOT, "final_index", "bm25.pkl")

text_index_df.to_parquet(text_index_path)
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)

print(f"Đã lưu bảng text : {text_index_path}")
print(f"Đã lưu BM25 index: {bm25_path}")

---
## Phần 9.5 — Kiểm tra chéo: video_id có ĐẦY ĐỦ ở cả 4 nguồn không? (MỚI)

**Giải thích:** "Bước 1 — Ingest & Validate" ở đầu notebook chỉ kiểm tra CÓ file zip hay không, không kiểm tra từng `video_id` có mặt đủ ở cả 4 nguồn (CLIP/Objects/Map-keyframes/Media-info). Đây chính là lý do lỗi "3 video thiếu keyframe" trước đây chỉ lộ ra khi chạy Online, không bị bắt sớm ở Offline. Cell dưới đọc lại 4 file output CUỐI CÙNG (đã ghi lên Drive) và đối chiếu tập `video_id`, in rõ ra video nào thiếu ở nguồn nào.


In [ ]:
print("=" * 60)
print("KIỂM TRA CHÉO: video_id có ĐẦY ĐỦ ở cả 4 nguồn không?")
print("=" * 60)

_clip_map_check = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet"))
_objects_check = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "objects", "objects_index.parquet"))
_map_kf_check = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet"))

_media_path_check = os.path.join(EXTRACTED_ROOT, "media_info_analysis", "media_full.csv")
_media_check = (pd.read_csv(_media_path_check) if os.path.exists(_media_path_check)
                else pd.DataFrame(columns=["video_id"]))

sources = {
    "CLIP mapping":  set(_clip_map_check["video_id"].unique()),
    "Objects":       set(_objects_check["video_id"].unique()),
    "Map-keyframes": set(_map_kf_check["video_id"].unique()),
    "Media-info":    set(_media_check["video_id"].unique()) if len(_media_check) else set(),
}

all_video_ids = set().union(*sources.values())
print(f"Tổng số video_id xuất hiện ở ÍT NHẤT 1 nguồn: {len(all_video_ids)}\n")

any_missing = False
for name, ids in sources.items():
    missing = all_video_ids - ids
    if missing:
        any_missing = True
    status = "OK — đủ" if not missing else f"THIẾU {len(missing)} video"
    print(f"[{status:14}] {name:16} ({len(ids)} video)")
    if missing:
        sample = sorted(missing)[:10]
        print(f"             ví dụ video bị thiếu: {sample}{' ...' if len(missing) > 10 else ''}")

if not any_missing:
    print("\n=> Tất cả video_id đồng nhất giữa 4 nguồn. Không có lỗ hổng dữ liệu kiểu '3 video thiếu keyframe'.")
else:
    print("\n=> CẢNH BÁO: có video bị thiếu ở ít nhất 1 nguồn — kiểm tra lại bước tương ứng phía trên")
    print("   trước khi build BM25/FAISS cuối cùng, vì các video này sẽ KHÔNG THỂ retrieval đúng.")


---
## Phần 10 — Bước 7: Consolidate — Tổng kết Index Store

**Giải thích:** in ra toàn bộ danh sách file kết quả cuối cùng trong `extracted/` — đây chính là "Index Store" mà Online Retrieval Service (Spring Boot + Python) sẽ tải về máy để chạy, không cần biết chi tiết 8 bước offline đã tạo ra nó thế nào.

In [ ]:
print("=" * 60)
print("INDEX STORE — TỔNG KẾT")
print("=" * 60)

final_files = {
    "CLIP FAISS Index":  os.path.join(EXTRACTED_ROOT, "clip_index", "clip_faiss.index"),
    "CLIP Mapping":      os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet"),
    "Objects Table":     os.path.join(EXTRACTED_ROOT, "objects", "objects_index.parquet"),
    "Media-info Table":  os.path.join(EXTRACTED_ROOT, "media_info_analysis", "media_full.csv"),
    "Map-keyframes":     os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet"),
    "Text Index (OCR+ASR)": os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet"),
    "BM25 Index":        os.path.join(EXTRACTED_ROOT, "final_index", "bm25.pkl"),
}

for label, path in final_files.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"[OK]    {label:24} {size_mb:8.2f} MB   {path}")
    else:
        print(f"[THIẾU] {label:24} — chưa tạo, kiểm tra lại các Phần phía trên")

# 2 file này KHÔNG được tạo bởi notebook Offline này — Online (Phần 1) vẫn cần chúng
# ở CÙNG thư mục media_info_analysis/. Kiểm tra RIÊNG, không lẫn vào bảng trên vì
# nguồn gốc khác hẳn ("chưa chạy phần này" khác với "notebook khác chưa chạy").
print()
worklist_files = {
    "worklist_kis_qa.csv": os.path.join(EXTRACTED_ROOT, "media_info_analysis", "worklist_kis_qa.csv"),
    "worklist_trake.csv":  os.path.join(EXTRACTED_ROOT, "media_info_analysis", "worklist_trake.csv"),
}
for label, path in worklist_files.items():
    status = "[OK]   " if os.path.exists(path) else "[THIẾU]"
    note = "" if os.path.exists(path) else "  <- tạo bởi notebook KHÁC (VD: Media_Info_Exploitation), không phải notebook này"
    print(f"{status} {label:24} {path}{note}")


---
## Phần 11 — Bước 8: Incremental Update — chuẩn bị sẵn cho Batch 2

**Giải thích:** hàm mẫu dưới đây minh họa cách MỞ RỘNG index hiện có khi Batch 2 về, mà KHÔNG cần chạy lại toàn bộ notebook từ đầu. Đây chỉ là khung sườn — cần bổ sung logic cụ thể tùy định dạng Batch 2 thực tế khi có (đặc biệt: nếu Batch 2 KHÔNG có sẵn Keyframes/CLIP/Objects như Batch 1, cần dùng code fallback tự trích xuất đã bàn ở phần thiết kế, chưa đưa vào notebook này).

In [ ]:
def add_video_to_clip_index(video_id, npy_path, existing_index_path, existing_mapping_path):
    """Thêm 1 video mới vào FAISS Index đã có, không rebuild lại từ đầu."""
    index = faiss.read_index(existing_index_path)
    mapping_df = pd.read_parquet(existing_mapping_path)

    arr = np.load(npy_path).astype("float32")
    faiss.normalize_L2(arr)
    index.add(arr)

    new_rows = pd.DataFrame([{"video_id": video_id, "local_frame_idx": i} for i in range(arr.shape[0])])
    mapping_df = pd.concat([mapping_df, new_rows], ignore_index=True)

    faiss.write_index(index, existing_index_path)
    mapping_df.to_parquet(existing_mapping_path)
    print(f"Đã thêm video {video_id} ({arr.shape[0]} vector) vào index hiện có.")

print("Hàm add_video_to_clip_index() sẵn sàng dùng khi Batch 2 về.")
print("Áp dụng nguyên tắc tương tự cho Objects, OCR, ASR: luôn dùng manifest để CHỈ xử lý video MỚI.")

---
## Phần 10 — ReCap Captioning: caption có trí nhớ ngữ cảnh xuyên suốt video

**Mục tiêu:** tạo caption chất lượng cao cho từng đoạn (shot) trong video, KHÁC OCR/ASR (đọc chữ/lời nói) — đây là mô tả HÌNH ẢNH bằng Gemini Vision, có "trí nhớ" (memory) mang từ shot trước sang shot sau trong CÙNG 1 video, giúp caption biết được bối cảnh (tên món, danh tính người, sự kiện đã xảy ra) dù bản thân shot đó không có gì trực tiếp thể hiện điều đó (VD: shot giữa video biết đây là "món bún gà" dù chỉ shot ĐẦU video nhắc tên món).

**Cách chia shot:** dùng độ tương đồng CLIP giữa 2 keyframe liên tiếp (đã có sẵn vector, không cần model mới) — ngưỡng tính RIÊNG cho từng video theo percentile (`SHOT_SPLIT_PERCENTILE`, mặc định P5), vì mỗi video có đặc tính chuyển động khác nhau (video tin tức khác video đua xe đạp) — dùng 1 số cố định chung cho mọi video (đã thử nghiệm với dữ liệu thật) cho kết quả sai lệch nhiều (44-78% cặp bị coi nhầm là chuyển cảnh), nên percentile-riêng-từng-video là cách đã được xác nhận đúng qua kiểm chứng thật.

**⚠️ CẢNH BÁO QUY MÔ — đọc trước khi chạy full:** ước tính ~13.000-25.000 lượt gọi Gemini cho 873 video (15-30 shot/video). Free tier Flash-Lite giới hạn ~1.000 request/NGÀY — **chắc chắn phải chạy trải dài nhiều ngày**, không xong trong 1 phiên. Checkpoint theo TỪNG VIDEO (không phải theo shot, vì memory chỉ có nghĩa trong 1 video — giống lý do ASR checkpoint theo video, nhưng ở đây nếu bị ngắt giữa chừng 1 video phải làm lại TOÀN BỘ video đó, không chỉ phần dở dang, vì memory là chuỗi phụ thuộc tuần tự).

**Đề xuất mạnh: chạy thử `LIMIT_ZIPS=1` trước** (vài chục video) để xem chất lượng + tốc độ thật, trước khi quyết định chạy full 873 video.


### 10.0 — Setup: kết nối Gemini API + cơ chế rate limit/retry/timeout

Offline chưa từng dùng Gemini trước đây (mọi việc khác đều chạy model cục bộ: EasyOCR, PhoWhisper, CLAP) — cell này port NGUYÊN cơ chế đã được kiểm chứng bên Online (rate limiter, retry có backoff, timeout 60s chặn treo vô thời hạn) để đảm bảo an toàn cho 1 việc chạy hàng chục nghìn lượt gọi trải dài nhiều ngày như ReCap.


In [ ]:
import time, random
from google import genai
from google.genai import types
from google.colab import userdata

GEMINI_API_KEY1 = userdata.get("GEMINI_API_KEY1")
gemini_client = genai.Client(api_key=GEMINI_API_KEY1)
GEMINI_MODEL = "gemini-3.5-flash-lite"   # bulk/số lượng lớn -> Flash-Lite, KHÔNG dùng Flash
                                          # (Flash chỉ 20 request/ngày free tier, xem Online)
GEMINI_TIMEOUT_MS = 60_000   # chặn treo vô thời hạn (bug SDK đã biết, xem Online Phần 13)

# FIX: tự động chuyển sang GEMINI_API_KEY2 (đã có sẵn từ setup Online, model Judge) khi
# key1 hết quota/NGÀY — key2 CHƯA TỪNG gọi model flash-lite trước đây (Online chỉ dùng key2
# cho model "flash" Judge), nên nhiều khả năng còn quota flash-lite hoàn toàn mới, chưa đụng.
try:
    GEMINI_API_KEY2 = userdata.get("GEMINI_API_KEY2")
    gemini_client_backup = genai.Client(api_key=GEMINI_API_KEY2) if GEMINI_API_KEY2 else None
except Exception:
    gemini_client_backup = None

_active_gemini_client = gemini_client   # biến toàn cục — đổi khi phát hiện hết quota/ngày


def get_active_gemini_client():
    return _active_gemini_client


def switch_to_backup_gemini_key():
    global _active_gemini_client
    if gemini_client_backup is not None and _active_gemini_client is not gemini_client_backup:
        print("    [CHUYỂN KEY] Key hiện tại có dấu hiệu hết quota/ngày -> chuyển sang GEMINI_API_KEY2")
        _active_gemini_client = gemini_client_backup
        return True
    return False


print("GEMINI_API_KEY2 (dự phòng):", "có sẵn" if gemini_client_backup is not None else "CHƯA CÓ — không tự chuyển key được khi hết quota")


class GeminiRateLimiter:
    def __init__(self, max_calls_per_minute: int = 14):
        self.max_calls = max_calls_per_minute
        self.call_times = []

    def wait(self):
        now = time.time()
        self.call_times = [t for t in self.call_times if now - t < 60]
        if len(self.call_times) >= self.max_calls:
            sleep_time = 60 - (now - self.call_times[0]) + 0.1
            print(f"    [RateLimiter] {self.max_calls} call trong 60s qua -> nghỉ {sleep_time:.1f}s")
            time.sleep(max(sleep_time, 0))
        self.call_times.append(time.time())


recap_rate_limiter = GeminiRateLimiter(max_calls_per_minute=14)


def call_gemini_with_retry(fn, max_retries: int = 5, limiter: "GeminiRateLimiter" = None):
    """fn: hàm KHÔNG NHẬN THAM SỐ, PHẢI tự gọi get_active_gemini_client() bên trong (không
    đóng cứng 1 client cố định) — để lần gọi lại sau khi switch_to_backup_gemini_key() dùng
    đúng key mới, không phải key cũ đã hết quota."""
    limiter = limiter or recap_rate_limiter
    for attempt in range(max_retries + 1):
        limiter.wait()
        try:
            return fn()
        except Exception as e:
            err_str = str(e).lower()
            is_quota_exhausted = "429" in err_str or "resource_exhausted" in err_str
            if is_quota_exhausted:
                # THỬ CHUYỂN KEY TRƯỚC — nếu là hết quota/NGÀY, chờ backoff (vài chục giây)
                # hoàn toàn vô ích, chỉ chuyển key mới thật sự giải quyết được ngay lập tức.
                if switch_to_backup_gemini_key():
                    continue   # thử lại NGAY với key mới, không cần chờ backoff
            is_retriable = (is_quota_exhausted or "503" in err_str
                            or "timeout" in err_str or "timed out" in err_str or "deadline" in err_str)
            if is_retriable:
                wait_time = (2 ** attempt) * 2 + random.random()
                print(f"    [Retry Gemini] lần {attempt+1}/{max_retries+1} sau {wait_time:.1f}s "
                      f"({e.__class__.__name__}: {str(e)[:100]})")
                time.sleep(wait_time)
                continue
            raise
    raise RuntimeError(f"Gemini thất bại sau {max_retries+1} lần thử (đã thử cả key dự phòng nếu có)")


print("Đã kết nối Gemini API cho ReCap — model:", GEMINI_MODEL)


### 10.1 — Load lại CLIP index + hàm chia shot theo percentile (đã kiểm chứng với data thật)

Load lại `clip_index`/`clip_mapping_df` từ Drive (không giả định vẫn còn trong bộ nhớ từ Phần 2 — Phần này có thể chạy ở phiên Colab khác hẳn, nhiều ngày sau).


In [ ]:
import faiss
import numpy as np

clip_index = faiss.read_index(os.path.join(EXTRACTED_ROOT, "clip_index", "clip_faiss.index"))
clip_mapping_df = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet"))
print(f"Đã load lại CLIP index: {clip_index.ntotal} vector, {clip_mapping_df['video_id'].nunique()} video")

SHOT_SPLIT_PERCENTILE = 5   # P5 mặc định (tiết kiệm quota hơn P10) — đổi thành 10 nếu muốn
                             # shot mịn hơn (đã kiểm chứng bằng data thật, xem thảo luận trước)


def compute_shot_boundaries(video_id: str, percentile: float = SHOT_SPLIT_PERCENTILE):
    """Chia keyframe của 1 video thành các shot — dựa vào độ tương đồng CLIP giữa 2 keyframe
    LIÊN TIẾP, KHÔNG dựa vào khoảng cách frame_idx (đã kiểm chứng: BTC trích keyframe theo
    khoảng thời gian gần như CỐ ĐỊNH, không theo mật độ chuyển động thật -> khoảng cách
    frame_idx không phải tín hiệu đáng tin). Ngưỡng tính RIÊNG cho từng video theo percentile
    (không dùng 1 số cố định chung — đã kiểm chứng: video khác loại nội dung có similarity
    nền rất khác nhau, VD đua xe đạp P10=0.72 trong khi tin tức P10=0.54).

    Trả về: list các shot, mỗi shot là dict {"local_frame_idxs": [...]} (danh sách
    local_frame_idx thuộc shot đó, đã sort theo thời gian).
    """
    rows = clip_mapping_df[clip_mapping_df["video_id"] == video_id].sort_values("local_frame_idx")
    if len(rows) == 0:
        return []
    if len(rows) == 1:
        return [{"local_frame_idxs": rows["local_frame_idx"].tolist()}]

    positions = rows.index.to_numpy()
    local_idxs = rows["local_frame_idx"].to_numpy()
    vectors = np.vstack([clip_index.reconstruct(int(p)) for p in positions]).astype("float32")
    faiss.normalize_L2(vectors)
    sims = (vectors[:-1] * vectors[1:]).sum(axis=1)   # similarity giữa mỗi cặp liên tiếp
    threshold = float(np.percentile(sims, percentile))

    shots = [{"local_frame_idxs": [int(local_idxs[0])]}]
    for i, sim in enumerate(sims):
        if sim < threshold:
            shots.append({"local_frame_idxs": []})   # bắt đầu shot mới
        shots[-1]["local_frame_idxs"].append(int(local_idxs[i + 1]))

    return shots


def find_video_keyframe_dir(local_dir: str, video_id: str):
    """Tìm thư mục ảnh của 1 video sau khi giải nén — dùng rglob (đệ quy MỌI cấp thư mục),
    KHÔNG giả định đúng 1 cấp cố định (local_dir/video_id/). FIX BUG THỰC TẾ ĐÃ GẶP: cấu
    trúc bên trong 1 số file zip có thêm 1 cấp thư mục lồng (VD Keyframes_L21/L21_V001/...)
    khiến Path(local_dir, video_id) không tồn tại -> mọi shot bị bỏ qua âm thầm, "0 shot"
    cho MỌI video mà không hề báo lỗi gì. Cách làm này khớp đúng cách OCR (Phần 5) đã dùng
    rglob thành công từ trước — chỉ là ReCap trước đó không theo đúng pattern đã có sẵn."""
    matches = [p for p in Path(local_dir).rglob(video_id) if p.is_dir()]
    return matches[0] if matches else None


def find_keyframe_image_path(video_dir, local_frame_idx: int):
    """video_dir: đường dẫn ĐÃ TÌM ĐƯỢC từ find_video_keyframe_dir() — không phải local_dir
    gốc. Dò theo GIÁ TRỊ SỐ của tên file (không giả định độ rộng đệm số 0 cố định, VD
    "001.jpg" hay "1.jpg" đều nhận diện được)."""
    if video_dir is None:
        return None
    n_target = local_frame_idx + 1   # công thức đã kiểm chứng: n = local_frame_idx + 1
    for img_path in Path(video_dir).glob("*.jpg"):
        try:
            if int(img_path.stem) == n_target:
                return str(img_path)
        except ValueError:
            continue
    return None


### 10.2 — Setup vòng lặp chính: load OCR/ASR để làm ngữ cảnh bổ sung + checkpoint


In [ ]:
LIMIT_ZIPS = None   # ĐÃ XÁC NHẬN chất lượng ổn qua chạy thử (L21 tin tức + L26 nấu ăn)
                     # -> chạy FULL toàn bộ 873 video. Nhớ: checkpoint theo VIDEO, an toàn để
                     # dừng/chạy lại nhiều lần qua nhiều ngày (xem cảnh báo quy mô ở trên).

recap_dest = os.path.join(EXTRACTED_ROOT, "recap_captions")
os.makedirs(recap_dest, exist_ok=True)
recap_manifest = os.path.join(recap_dest, "_manifest.txt")
done_recap_videos = load_manifest(recap_manifest)

keyframe_zips = sorted(Path(DATASET_ROOT, "keyframes").glob("*.zip"))
if LIMIT_ZIPS:
    keyframe_zips = keyframe_zips[:LIMIT_ZIPS]

# Load OCR/ASR đã có (Phần 9) làm ngữ cảnh BỔ SUNG cho mỗi shot — không bắt buộc phải có,
# nếu thiếu vẫn chạy được (chỉ mất phần ngữ cảnh chữ/lời nói, còn ảnh + memory vẫn hoạt động)
text_index_path = os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet")
text_index_df = pd.read_parquet(text_index_path) if os.path.exists(text_index_path) else pd.DataFrame()
map_keyframes_df_full = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet"))

print(f"Sẽ xử lý {len(keyframe_zips)} zip keyframes (đã xong trước đó: {len(done_recap_videos)} video)")
print(f"text_index_df: {len(text_index_df)} dòng {'(có sẵn)' if len(text_index_df) else '(CHƯA CÓ — chạy Phần 9 trước nếu muốn có ngữ cảnh OCR/ASR)'}")


def get_ocr_asr_context(video_id: str, shot_local_frame_idxs: list) -> str:
    """Lấy OCR/ASR trong đúng khoảng thời gian của shot này, làm ngữ cảnh bổ sung cho Gemini
    (best-effort — không có cũng không sao, chỉ làm caption bớt chính xác hơn 1 chút)."""
    if len(text_index_df) == 0:
        return "không có"
    n_values = [i + 1 for i in shot_local_frame_idxs]   # local_frame_idx -> n (1-based)

    video_map = map_keyframes_df_full[map_keyframes_df_full["video_id"] == video_id]
    pts_in_shot = video_map[video_map["n"].isin(n_values)]["pts_time"]
    t_start = float(pts_in_shot.min()) if len(pts_in_shot) else 0.0
    t_end = float(pts_in_shot.max()) if len(pts_in_shot) else 0.0

    video_text = text_index_df[text_index_df["video_id"] == video_id]
    ocr_rows = video_text[(video_text["source"] == "OCR") &
                          (pd.to_numeric(video_text["frame_index"], errors="coerce").isin(n_values))]
    asr_rows = video_text[(video_text["source"] == "ASR") &
                          (video_text["start"] <= t_end) & (video_text["end"] >= t_start)]

    parts = []
    if len(ocr_rows):
        parts.append("Chữ trên màn hình: " + " | ".join(ocr_rows["text"].dropna().unique()[:3]))
    if len(asr_rows):
        parts.append("Lời nói: " + " ".join(asr_rows["text"].dropna().tolist()[:3]))
    return " ; ".join(parts) if parts else "không có"


### 10.3 — Vòng lặp chính: chia shot -> duyệt tuần tự có bộ nhớ -> lưu theo video


In [ ]:
def recap_one_video(video_id: str, local_dir: str) -> list:
    """Xử lý ĐÚNG 1 video: chia shot, duyệt TUẦN TỰ (không song song — memory phụ thuộc
    thứ tự), trả về list các dòng caption. Nếu bị lỗi giữa chừng, ném exception ra ngoài để
    vòng lặp cha quyết định KHÔNG mark_done (video coi như chưa xong, thử lại từ đầu sau)."""
    shots = compute_shot_boundaries(video_id)
    if not shots:
        return []

    video_dir = find_video_keyframe_dir(local_dir, video_id)
    if video_dir is None:
        print(f"    [CẢNH BÁO] {video_id}: không tìm thấy thư mục ảnh sau khi giải nén "
              f"(kiểm tra lại cấu trúc zip nếu vẫn 0 shot sau fix rglob)")
        return []

    memory = ""   # RỖNG khi bắt đầu — reset cho mỗi video
    records = []

    for shot_idx, shot in enumerate(shots):
        mid_local_idx = shot["local_frame_idxs"][len(shot["local_frame_idxs"]) // 2]
        img_path = find_keyframe_image_path(video_dir, mid_local_idx)
        if img_path is None:
            continue   # bỏ qua shot này nếu không tìm được ảnh đại diện, không dừng cả video

        context_text = get_ocr_asr_context(video_id, shot["local_frame_idxs"])

        with open(img_path, "rb") as f:
            image_bytes = f.read()

        prompt = f"""Bối cảnh đã biết từ các đoạn TRƯỚC trong video này: {memory or "(chưa có gì, đây là đoạn đầu tiên)"}
Ngữ cảnh chữ/lời nói tại đoạn này: {context_text}

Nhiệm vụ:
1. Mô tả CHI TIẾT những gì đang diễn ra trong ảnh này, TẬN DỤNG bối cảnh đã biết ở trên
   nếu liên quan (VD nếu đã biết tên món ăn/tên người/sự kiện từ trước, hãy nhắc lại nếu
   ảnh này vẫn thuộc cùng bối cảnh đó).
2. Cập nhật lại bối cảnh (memory) cho ĐOẠN SAU — giữ thông tin còn quan trọng (tên món/
   người/sự kiện/địa điểm), bỏ chi tiết đã lỗi thời, tối đa khoảng 80 từ.

Trả về ĐÚNG 1 JSON: {{"caption": "<mô tả chi tiết>", "memory": "<bối cảnh cập nhật>"}}"""

        def _call():
            response = get_active_gemini_client().models.generate_content(
                model=GEMINI_MODEL,
                contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"), prompt],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_MS),
                ),
            )
            return json.loads(response.text)

        result = call_gemini_with_retry(_call)

        records.append({
            "video_id": video_id,
            "shot_idx": shot_idx,
            "start_local_frame_idx": shot["local_frame_idxs"][0],
            "end_local_frame_idx": shot["local_frame_idxs"][-1],
            "caption": result.get("caption", ""),
            "source": "RECAP",
        })
        memory = result.get("memory", memory)   # mang sang shot TIẾP THEO

    return records


for zip_path in keyframe_zips:
    zip_name = zip_path.name

    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        video_ids_in_zip = sorted(set(Path(n).parent.name for n in zf.namelist() if n.lower().endswith(".jpg")))

    if video_ids_in_zip and all(vid in done_recap_videos for vid in video_ids_in_zip):
        print(f"[SKIP] {zip_name} — tất cả {len(video_ids_in_zip)} video đã xử lý xong trước đó")
        continue

    print(f"[BẮT ĐẦU] {zip_name} — {len(video_ids_in_zip)} video")
    local_dir = os.path.join(LOCAL_TMP, "recap_kf_temp")
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    os.makedirs(local_dir)
    with zipfile.ZipFile(str(zip_path), 'r') as zf:
        zf.extractall(local_dir)

    videos_to_process = [v for v in video_ids_in_zip if v not in done_recap_videos]
    print(f"  Cần xử lý: {len(videos_to_process)}/{len(video_ids_in_zip)} video")

    for video_id in videos_to_process:
        try:
            records = recap_one_video(video_id, local_dir)
            out_path = os.path.join(recap_dest, f"recap_{video_id}.parquet")
            pd.DataFrame(records).to_parquet(out_path)
            mark_done(recap_manifest, video_id)
            done_recap_videos.add(video_id)
            print(f"  [XONG] {video_id}: {len(records)} shot")
        except Exception as e:
            print(f"  [LỖI] {video_id}: {e} -> KHÔNG mark_done, sẽ thử lại lần chạy sau")

    print(f"[XONG ZIP] {zip_name}")
    shutil.rmtree(local_dir)


### 10.4 — Kiểm tra ReCap đã chạy đủ và đúng chưa (trước khi gộp vào index chính)


In [ ]:
all_video_ids = set(clip_mapping_df["video_id"].unique())
done_recap_videos_final = load_manifest(recap_manifest)
missing = all_video_ids - done_recap_videos_final

print(f"Tổng số video trong dataset : {len(all_video_ids)}")
print(f"Đã xong ReCap               : {len(done_recap_videos_final)}")
print(f"CÒN THIẾU                   : {len(missing)}")
if missing:
    print(f"  Danh sách thiếu (tối đa 20): {sorted(missing)[:20]}")
    print("  -> Chạy lại Phần 10.3 (Run all từ đó) để tiếp tục xử lý nốt các video còn thiếu.")

# Kiểm tra video có 0 shot bất thường (nghi ngờ lỗi, không phải video thật sự ít shot)
recap_files = list(Path(recap_dest).glob("recap_*.parquet"))
empty_videos = []
shot_counts = []
for p in recap_files:
    df = pd.read_parquet(p)
    shot_counts.append(len(df))
    if len(df) == 0:
        empty_videos.append(p.stem.replace("recap_", ""))

print(f"\nSố shot trung bình/video: {sum(shot_counts)/max(len(shot_counts),1):.1f}")
print(f"Video có 0 shot (đáng ngờ): {len(empty_videos)}")
if empty_videos:
    print(f"  Danh sách (tối đa 20): {empty_videos[:20]}")
    print("  -> Các video này cần xóa checkpoint riêng rồi chạy lại — hỏi Claude nếu số lượng nhiều.")


### 10.5 — Gộp ReCap vào `text_index.parquet` + build lại BM25

Quy đổi `local_frame_idx` (0-based, CLIP) → frame_id thật (cùng công thức `n = local_frame_idx + 1` đã kiểm chứng nhiều lần trong suốt dự án) — lưu vào 2 cột MỚI `start_frame`/`end_frame` (KHÔNG dùng chung cột `start`/`end` của ASR, vì ASR lưu đơn vị THỜI GIAN (giây) còn RECAP là SỐ FRAME — trộn lẫn đơn vị trong cùng 1 cột dễ gây lỗi khó phát hiện về sau). Cũng lưu `frame_index` = frame ĐẠI DIỆN (frame giữa shot, khớp đúng ảnh đã dùng để sinh caption) để tương thích với cách OCR đang dùng (1 frame/dòng), phòng trường hợp Online muốn dùng RECAP như OCR mà chưa cần sửa gì thêm.

**Chạy Run all lại từ Phần 9 (BM25) trước khi chạy cell này** — cần biến `text_index_df` gốc (OCR+ASR) đã có trong bộ nhớ. Nếu chạy ở phiên mới, cell sẽ tự load lại từ Drive.


In [ ]:
def get_real_frame_id_offline(video_id: str, local_frame_idx):
    """Bản Offline của get_real_frame_id() — công thức n = local_frame_idx + 1 đã kiểm
    chứng kỹ nhiều lần trong dự án (test cosine=1.0, đối chiếu độ lệch offset 0/±5/±10)."""
    if local_frame_idx is None or pd.isna(local_frame_idx):
        return None
    n = int(local_frame_idx) + 1
    row = map_keyframes_df_full[(map_keyframes_df_full["video_id"] == video_id) & (map_keyframes_df_full["n"] == n)]
    return int(row.iloc[0]["frame_idx"]) if len(row) else None


# Load lại text_index_df gốc (OCR+ASR) nếu chưa có trong bộ nhớ (phiên mới)
if "text_index_df" not in dir() or text_index_df is None:
    text_index_df = pd.read_parquet(os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet"))
    print(f"Đã load lại text_index_df gốc: {len(text_index_df)} dòng")

# Gộp toàn bộ file ReCap nhỏ lẻ, giống hệt cách OCR/ASR đã gộp ở Phần 9
# FIX BUG THỰC TẾ ĐÃ GẶP: loại trừ chính file kết quả cuối ("recap_index.parquet") khỏi
# danh sách part cần gộp — vì tên nó cũng khớp pattern "recap_*.parquet", nếu cell này đã
# từng chạy 1 lần trước đó, lần chạy sau sẽ tự gộp NHẦM CHÍNH KẾT QUẢ CŨ vào làm 1 "part",
# gây trùng cột (file cũ đã có cột "text", file per-video vẫn còn "caption" -> sau rename
# thành 2 cột "text" trùng tên).
recap_parts = [pd.read_parquet(p) for p in Path(recap_dest).glob("recap_*.parquet")
               if p.name != "recap_index.parquet"]
recap_all = pd.concat(recap_parts, ignore_index=True) if recap_parts else pd.DataFrame()

if len(recap_all):
    # FIX BUG (đã gặp thật — ArrowTypeError khi ghi parquet): "frame_index" của OCR lưu dạng
    # CHUỖI (VD "001" — số thứ tự keyframe n), khác hẳn Ý NGHĨA lẫn KIỂU DỮ LIỆU so với con số
    # RECAP tính ra (frame_idx THẬT, số nguyên, VD 270). Gán chung vào 1 cột "frame_index" vừa
    # gây lỗi kiểu dữ liệu khi ghi parquet (trộn str + int), vừa có nguy cơ đọc nhầm đơn vị nếu
    # Online xử lý RECAP giống hệt cách xử lý OCR. Để trống frame_index cho RECAP (giống cách
    # ASR cũng để trống — ASR vốn cũng không dùng frame_index, chỉ dùng start/end) — dùng riêng
    # start_frame/end_frame (đặt tên RÕ RÀNG, không lẫn với cột nào của OCR/ASR).
    recap_all["frame_index"] = None
    recap_all["start_frame"] = recap_all.apply(
        lambda r: get_real_frame_id_offline(r["video_id"], r["start_local_frame_idx"]), axis=1)
    recap_all["end_frame"] = recap_all.apply(
        lambda r: get_real_frame_id_offline(r["video_id"], r["end_local_frame_idx"]), axis=1)
    recap_all = recap_all.rename(columns={"caption": "text"})
    recap_all = recap_all[["video_id", "frame_index", "start_frame", "end_frame", "text", "source"]]

    # Lưu RIÊNG 1 bản recap_index.parquet (độc lập, dễ kiểm tra/debug sau này)
    recap_index_path = os.path.join(recap_dest, "recap_index.parquet")
    recap_all.to_parquet(recap_index_path)
    print(f"Đã lưu: {recap_index_path} ({len(recap_all)} dòng)")

    # Gộp vào text_index_df chính (OCR + ASR + RECAP)
    text_index_df = pd.concat([text_index_df, recap_all], ignore_index=True)
    print(f"\ntext_index_df sau khi gộp: {len(text_index_df)} dòng")
    print(text_index_df["source"].value_counts())

    # Build lại BM25 với corpus đã có thêm RECAP
    tokenized_corpus = [str(t).lower().split() for t in text_index_df["text"].fillna("")]
    bm25 = BM25Okapi(tokenized_corpus)

    text_index_df.to_parquet(text_index_path)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)
    print(f"\nĐã lưu lại: {text_index_path}")
    print(f"Đã lưu lại: {bm25_path}")
else:
    print("Không tìm thấy file recap_*.parquet nào — kiểm tra lại đã chạy Phần 10.3 chưa.")
